# Clean ATP UK Data

Load and clean UK Data

## Set Up Notebook

In [272]:
# Import Packages
from pathlib import Path
import pandas as pd
import re


# Import Local Package
from tennis_data_pipeline.validatation.tennis_data_uk.tournaments import (
    find_uk_inconsistent_tournaments, 
    find_uk_reused_tournament_ids, 
)
from tennis_data_pipeline.cleaner.uk import atp_cols as cols
from tennis_data_pipeline.cleaner.uk.atp import(
    build_uk_atp_quality_report,
    clean_uk_atp_data,
    summarize_uk_atp_quality,
)


# Constants
YEAR = 2016
# YEAR = int(input("Enter the year: "))
PROJECT_DIR = Path("../..").resolve()
print("Base Directory:", PROJECT_DIR)


Base Directory: /Users/nicholasbenelli/Workspace/repos/GitHub/-sports/-tennis/Tennis-Data-Pipeline


### Constants

In [273]:
def load_dirty_uk_atp_data(year: int) -> pd.DataFrame:
    df_uk = pd.read_csv(PROJECT_DIR / f"data/raw/tennis-data-uk/2024-10/atp/atp_singles_results_{year}.csv")

    # Date format drifts across seasons (e.g. "1/1/23" vs. "2000-01-03").
    df_uk["Date"] = pd.to_datetime(df_uk["Date"], format="mixed", dayfirst=False)
    df_uk["Year"] = year

    # Nullable ints: ranks/points/set-scores are whole numbers but can be missing (e.g. retired matches).
    int_cols = [
        "ATP", "Year", "Best of",
        "WRank", "LRank", "WPts", "LPts",
        "W1", "L1", "W2", "L2", "W3", "L3", "W4", "L4", "W5", "L5",
        "Wsets", "Lsets",
    ]
    for col in int_cols:
        if col in df_uk.columns:
            df_uk[col] = pd.to_numeric(df_uk[col], errors="coerce").astype("Int64")

    # Everything left over is bookmaker odds; which bookmakers are present varies by year.
    known_cols = {
        "ATP", "Year", "Location", "Tournament", "Date", "Series", "Court",
        "Surface", "Round", "Best of", "Winner", "Loser", "Comment", *int_cols,
    }
    odds_cols = [col for col in df_uk.columns if col not in known_cols]
    df_uk[odds_cols] = df_uk[odds_cols].apply(pd.to_numeric, errors="coerce")

    for col in ["Series", "Court", "Surface", "Round", "Comment"]:
        if col in df_uk.columns:
            df_uk[col] = df_uk[col].astype("category")

    return df_uk


## Load Data

In [274]:
df_uk_raw = load_dirty_uk_atp_data(YEAR)
df_uk = df_uk_raw.copy()
print(list(df_uk.columns))

print(df_uk.head())


['ATP', 'Location', 'Tournament', 'Date', 'Series', 'Court', 'Surface', 'Round', 'Best of', 'Winner', 'Loser', 'WRank', 'LRank', 'WPts', 'LPts', 'W1', 'L1', 'W2', 'L2', 'W3', 'L3', 'W4', 'L4', 'W5', 'L5', 'Wsets', 'Lsets', 'Comment', 'B365W', 'B365L', 'EXW', 'EXL', 'LBW', 'LBL', 'PSW', 'PSL', 'MaxW', 'MaxL', 'AvgW', 'AvgL', 'Year']
   ATP  Location              Tournament       Date  Series    Court Surface  \
0    1  Brisbane  Brisbane International 2016-01-04  ATP250  Outdoor    Hard   
1    1  Brisbane  Brisbane International 2016-01-04  ATP250  Outdoor    Hard   
2    1  Brisbane  Brisbane International 2016-01-04  ATP250  Outdoor    Hard   
3    1  Brisbane  Brisbane International 2016-01-04  ATP250  Outdoor    Hard   
4    1  Brisbane  Brisbane International 2016-01-05  ATP250  Outdoor    Hard   

       Round  Best of       Winner  ...   EXL   LBW   LBL   PSW   PSL  MaxW  \
0  1st Round        3  Dimitrov G.  ...  2.10  1.62  2.25  1.68  2.31  1.76   
1  1st Round        3     K

## Check Tournaments

In [275]:
ATP_BEST_OF_5_TOURNAMENTS = {
    "Australian Open",
    "French Open",
    "Roland Garros",
    "Wimbledon",
    "US Open",
}

# All ATP Grand Slams are best-of-5; the raw source mislabels some individual matches.
grand_slam_mask = df_uk["Tournament"].isin(ATP_BEST_OF_5_TOURNAMENTS)
df_uk.loc[grand_slam_mask, "Best of"] = 5

# ATP Finals (Masters Cup) is best-of-3 for every round; the raw source omits
# "Best of" entirely for these rows rather than mislabeling it.
masters_cup_mask = df_uk["Series"] == "Masters Cup"
df_uk.loc[masters_cup_mask & df_uk["Best of"].isna(), "Best of"] = 3


In [276]:
inconsistent_tourneys = find_uk_inconsistent_tournaments(df_uk, key_columns=["ATP", "Year", "Location"], info_cols=["Tournament", "Series", "Court", "Surface", "Best of"])
print("Inconsistent Tournaments:")
print(inconsistent_tourneys[0])

print("Rows Effected")
display(inconsistent_tourneys[1])

if not inconsistent_tourneys[0].empty and not inconsistent_tourneys[1].empty:
    raise ValueError("There are inconsistent tournaments with no affected rows.")

All tournament attributes are consistent.
Inconsistent Tournaments:
Empty DataFrame
Columns: [Tournament, Series, Court, Surface, Best of]
Index: []
Rows Effected


,ATP,Location,Tournament,Date,Series,Court,Surface,Round,Best of,Winner,...,EXL,LBW,LBL,PSW,PSL,MaxW,MaxL,AvgW,AvgL,Year


In [277]:
reused_tourney_id = find_uk_reused_tournament_ids(df=df_uk, id_col= "ATP", disambiguating_cols=["Location", "Tournament"])
print("ATP Reused Tournament IDs")
display(reused_tourney_id[0])

print("Rows Affected")
display(reused_tourney_id[1])

if not reused_tourney_id[0].empty and not reused_tourney_id[1].empty:
    raise ValueError("There are reused tournament IDs with no affected rows.")

Every ATP id maps to exactly one tournament.
ATP Reused Tournament IDs


,Location,Tournament
ATP,,


Rows Affected


,ATP,Location,Tournament,Date,Series,Court,Surface,Round,Best of,Winner,...,EXL,LBW,LBL,PSW,PSL,MaxW,MaxL,AvgW,AvgL,Year


In [278]:
# df_uk.loc[df_uk["Location"] == "Montpellier"]

### Bad Odds Checker

In [279]:
RAW_ODDS_COLS = ["B365W", "B365L", "PSW", "PSL", "MaxW", "MaxL", "AvgW", "AvgL"]

# Decimal odds are always >= 1.0; the raw source has occasional entry errors (e.g. a
# misplaced decimal point) that produce an impossible odd below 1. Capture the
# original rows for review before nulling them out (rather than guessing the
# intended value).
bad_odds_mask = (df_uk[RAW_ODDS_COLS] < 1).any(axis=1)
bad_odds_rows = df_uk.loc[
    bad_odds_mask,
    ["ATP", "Location", "Tournament", "Round", "Winner", "Loser", "Comment", *RAW_ODDS_COLS],
]

for col in RAW_ODDS_COLS:
    if col in df_uk.columns:
        bad_odds = df_uk[col] < 1
        if bad_odds.any():
            print(f"{col}: nulling {bad_odds.sum()} odds < 1")
            df_uk.loc[bad_odds, col] = pd.NA

print(bad_odds_rows)

Empty DataFrame
Columns: [ATP, Location, Tournament, Round, Winner, Loser, Comment, B365W, B365L, PSW, PSL, MaxW, MaxL, AvgW, AvgL]
Index: []


In [280]:
print(f"{len(bad_odds_rows)} row(s) had an odd below 1.0 (nulled above):")
display(bad_odds_rows)


0 row(s) had an odd below 1.0 (nulled above):


,ATP,Location,Tournament,Round,Winner,Loser,Comment,B365W,B365L,PSW,PSL,MaxW,MaxL,AvgW,AvgL


### Bad Set-Score Checker

In [281]:
if YEAR == 2019:
    # 2019 Metz final (Tsonga d. Bedene): raw source has a corrupted set-2 score and
    # is missing set 3 entirely, so Wsets/Lsets (0-1) contradict the "Completed" status.
    # Verified actual result was 6-7, 7-6, 6-3 to Tsonga (en.wikipedia.org/wiki/2019_Moselle_Open).
    metz_final_mask = (
        (df_uk["ATP"] == 53)
        & (df_uk["Winner"] == "Tsonga J.W.")
        & (df_uk["Loser"] == "Bedene A.")
        & (df_uk["Date"] == "2019-09-22")
    )
    print(f"Rows matched: {metz_final_mask.sum()}")

    df_uk.loc[metz_final_mask, ["W1", "L1", "W2", "L2", "W3", "L3"]] = [6, 7, 7, 6, 6, 3]
    df_uk.loc[metz_final_mask, ["Wsets", "Lsets"]] = [2, 1]

if YEAR == 2015:
    # 2015 Nottingham (AEGON Open) semifinal (Istomin d. Baghdatis): Wikipedia lists
    # Baghdatis under the tournament's retirements, but the raw source labels this
    # "Completed" despite only a single incomplete set (1-2) being recorded.
    nottingham_sf_mask = (
        (df_uk["ATP"] == 38)
        & (df_uk["Winner"] == "Istomin D.")
        & (df_uk["Loser"] == "Baghdatis M.")
        & (df_uk["Date"] == "2015-06-26")
    )
    print(f"Rows matched: {nottingham_sf_mask.sum()}")

    df_uk.loc[nottingham_sf_mask, "Comment"] = "Retired"

if YEAR == 2013:
    # 2013 Bogota (Claro Open Colombia) final (Karlovic d. Falla): raw source has
    # Wsets/Lsets both recorded as 0 despite two straight sets being recorded.
    # Verified actual result was 6-3, 7-6 to Karlovic (en.wikipedia.org/wiki/2013_Claro_Open_Colombia).
    bogota_final_mask = (
        (df_uk["ATP"] == 41)
        & (df_uk["Winner"] == "Karlovic I.")
        & (df_uk["Loser"] == "Falla A.")
        & (df_uk["Date"] == "2013-07-21")
    )
    print(f"Rows matched: {bogota_final_mask.sum()}")

    df_uk.loc[bogota_final_mask, ["Wsets", "Lsets"]] = [2, 0]


## Clean Data

In [282]:
df_uk = clean_uk_atp_data(df_uk)

Sanity-check renamed categories

In [283]:
for col in [
    "series",
    "indoor_outdoor",
    "surface",
    "round",
    "match_status",
]:
    print(f"\n{col}")
    print(
        df_uk[col]
        .value_counts(dropna=False)
        .sort_index()
    )


series
series
atp_250         1054
atp_500          482
grand_slam       508
masters_1000     567
tour_finals       15
Name: count, dtype: int64

indoor_outdoor
indoor_outdoor
indoor      425
outdoor    2201
Name: count, dtype: int64

surface
surface
clay      806
grass     317
hard     1503
Name: count, dtype: int64

round
round
R128    1172
R64      752
R32      184
R16       48
QF       260
RR        12
SF       132
F         66
Name: count, dtype: int64

match_status
match_status
completed    2527
retired        82
walkover       17
Name: count, dtype: int64


## Explore Data Quality

In [284]:
summarize_uk_atp_quality(df_uk)

Rows: 2626
Duplicate match keys: 0

Match status:
match_status
completed    2527
retired        82
walkover       17
Name: count, dtype: int64

Missingness:
winner_set_5_games      96.496573
loser_set_5_games       96.496573
loser_set_4_games       90.517898
winner_set_4_games      90.517898
winner_set_3_games      53.236862
loser_set_3_games       53.236862
loser_set_2_games        1.523229
winner_set_2_games       1.523229
loser_sets               0.761615
winner_sets              0.761615
winner_set_1_games       0.685453
loser_set_1_games        0.685453
loser_rank               0.266565
loser_rank_points        0.266565
odds_pinnacle_winner     0.228484
odds_pinnacle_loser      0.228484
odds_b365_loser          0.114242
odds_b365_winner         0.114242
odds_avg_loser           0.076161
odds_max_winner          0.076161
odds_avg_winner          0.076161
odds_max_loser           0.076161
winner_rank              0.038081
winner_rank_points       0.038081
dtype: float64

Completed m

In [285]:
score_cols = [
    "winner_set_1_games",
    "loser_set_1_games",
    "winner_set_2_games",
    "loser_set_2_games",
    "winner_sets",
    "loser_sets",
]

for col in score_cols:
    print(f"\n{col}")
    print(
        df_uk.loc[df_uk[col].isna(), "match_status"]
        .value_counts(dropna=False)
    )


winner_set_1_games
match_status
walkover     17
retired       1
completed     0
Name: count, dtype: int64

loser_set_1_games
match_status
walkover     17
retired       1
completed     0
Name: count, dtype: int64

winner_set_2_games
match_status
retired      23
walkover     17
completed     0
Name: count, dtype: int64

loser_set_2_games
match_status
retired      23
walkover     17
completed     0
Name: count, dtype: int64

winner_sets
match_status
walkover     17
retired       3
completed     0
Name: count, dtype: int64

loser_sets
match_status
walkover     17
retired       3
completed     0
Name: count, dtype: int64


In [286]:
missing_first_set = df_uk.loc[
    df_uk["winner_set_1_games"].isna()
    | df_uk["loser_set_1_games"].isna(),
    [
        "match_date",
        "tournament_name",
        "round",
        "winner_name",
        "loser_name",
        "match_status",
        "winner_set_1_games",
        "loser_set_1_games",
        "winner_sets",
        "loser_sets",
    ],
]

print(missing_first_set)

     match_date                               tournament_name round  \
337  2016-02-04                       Garanti Koza Sofia Open   R64   
516  2016-02-19                                      Rio Open    QF   
658  2016-03-13                              BNP Paribas Open   R64   
737  2016-03-25                            Sony Ericsson Open   R64   
842  2016-04-06                          Grand Prix Hassan II   R64   
1167 2016-05-13                   Internazionali BNL d'Italia    QF   
1332 2016-05-28                                   French Open   R32   
1425 2016-06-15                              Gerry Weber Open   R64   
1435 2016-06-17                              Gerry Weber Open    QF   
1458 2016-06-16                           AEGON Championships   R64   
1600 2016-06-30                                     Wimbledon   R64   
2007 2016-08-18    Western & Southern Financial Group Masters   R32   
2040 2016-08-23  Winston-Salem Open at Wake Forest University   R64   
2139 2

In [287]:
missing_odds = df_uk.loc[
    df_uk[cols.ODDS_COLS].isna().any(axis=1),
    [
        "match_date",
        "tournament_name",
        "round",
        "winner_name",
        "loser_name",
        "match_status",
        *cols.ODDS_COLS,
    ],
]

print(missing_odds)

     match_date                               tournament_name round  \
1145 2016-05-10                   Internazionali BNL d'Italia   R64   
1424 2016-06-15                              Gerry Weber Open  R128   
1737 2016-07-19                            Suisse Open Gstaad  R128   
1922 2016-08-02                             BB&T Atlanta Open  R128   
2040 2016-08-23  Winston-Salem Open at Wake Forest University   R64   
2288 2016-09-28                                 Shenzhen Open  R128   

        winner_name     loser_name match_status  odds_b365_winner  \
1145     Pouille L.      Gulbis E.    completed              1.66   
1424       Thiem D.       Sousa J.    completed              1.22   
1737    Zeballos H.     Velotti A.    completed               NaN   
1922       Kamke T.  Stakhovsky S.    completed              2.50   
2040  Kuznetsov An.     Youzhny M.     walkover               NaN   
2288    Fabbiano T.    Nishioka Y.    completed               NaN   

      odds_b365_lo

## Quality Report

In [288]:
quality_report = build_uk_atp_quality_report(df_uk)
print(quality_report.T)


                                  Metric_2016
year                              2016.000000
rows                              2626.000000
duplicate_match_keys                 0.000000
status_count_completed            2527.000000
status_count_retired                82.000000
status_count_walkover               17.000000
completed_missing_odds               5.000000
missing_pct_winner_rank              0.038081
missing_pct_loser_rank               0.266565
missing_pct_winner_rank_points       0.038081
missing_pct_loser_rank_points        0.266565
missing_pct_winner_sets              0.761615
missing_pct_loser_sets               0.761615
missing_pct_winner_set_1_games       0.685453
missing_pct_loser_set_1_games        0.685453
missing_pct_winner_set_2_games       1.523229
missing_pct_loser_set_2_games        1.523229
missing_pct_winner_set_3_games      53.236862
missing_pct_loser_set_3_games       53.236862
missing_pct_winner_set_4_games      90.517898
missing_pct_loser_set_4_games     

In [289]:
quality_report.T

,Metric_2016
year,2016.000000
rows,2626.000000
duplicate_match_keys,0.000000
status_count_completed,2527.000000
status_count_retired,82.000000
status_count_walkover,17.000000
completed_missing_odds,5.000000
missing_pct_winner_rank,0.038081
missing_pct_loser_rank,0.266565
missing_pct_winner_rank_points,0.038081


In [295]:
# Write to quality report
quality_report_path = (
    PROJECT_DIR
    / "data/clean/tennis-data-uk/atp/analysis/uk_atp_quality_report.csv"
)
quality_report_path.parent.mkdir(parents=True, exist_ok=True)

# Accumulate one row per year across notebook runs;
# re-running a year overwrites its old row.
if quality_report_path.exists():
    existing_report = pd.read_csv(
        quality_report_path,
        index_col=0,
    )

    combined_report = pd.concat(
        [existing_report, quality_report]
    )

    combined_report = combined_report[
        ~combined_report.index.duplicated(keep="last")
    ]
else:
    combined_report = quality_report


# Metrics where a missing value really means "zero"
zero_fill_cols = [
    col
    for col in combined_report.columns
    if (
        col.startswith("status_count_")
        or col.startswith("missing_pct_")
        or col
        in {
            "duplicate_match_keys",
            "completed_missing_odds",
        }
    )
]

combined_report[zero_fill_cols] = (
    combined_report[zero_fill_cols]
    .fillna(0)
)

combined_report = combined_report.sort_values("year")
combined_report = combined_report.round(4)

combined_report.index.name = "index"

combined_report.to_csv(quality_report_path)

print(f"Written to {quality_report_path}")

Written to /Users/nicholasbenelli/Workspace/repos/GitHub/-sports/-tennis/Tennis-Data-Pipeline/data/clean/tennis-data-uk/atp/analysis/uk_atp_quality_report.csv


In [296]:
print(list(df_uk.columns))

print(df_uk.head())

['source', 'tour', 'year', 'uk_tournament_id', 'tournament_name', 'location', 'match_date', 'series', 'indoor_outdoor', 'surface', 'round', 'best_of', 'winner_name', 'loser_name', 'winner_rank', 'loser_rank', 'winner_rank_points', 'loser_rank_points', 'winner_sets', 'loser_sets', 'winner_set_1_games', 'loser_set_1_games', 'winner_set_2_games', 'loser_set_2_games', 'winner_set_3_games', 'loser_set_3_games', 'winner_set_4_games', 'loser_set_4_games', 'winner_set_5_games', 'loser_set_5_games', 'match_status', 'odds_b365_winner', 'odds_b365_loser', 'odds_pinnacle_winner', 'odds_pinnacle_loser', 'odds_max_winner', 'odds_max_loser', 'odds_avg_winner', 'odds_avg_loser', 'source_event_key', 'source_match_key']
           source tour  year  uk_tournament_id         tournament_name  \
0  tennis_data_uk  atp  2016                 1  Brisbane International   
1  tennis_data_uk  atp  2016                 1  Brisbane International   
2  tennis_data_uk  atp  2016                 1  Brisbane Internati

In [297]:
display(df_uk)

,source,tour,year,uk_tournament_id,tournament_name,location,match_date,series,indoor_outdoor,surface,...,odds_b365_winner,odds_b365_loser,odds_pinnacle_winner,odds_pinnacle_loser,odds_max_winner,odds_max_loser,odds_avg_winner,odds_avg_loser,source_event_key,source_match_key
0,tennis_data_uk,atp,2016,1,Brisbane International,Brisbane,2016-01-04,atp_250,outdoor,hard,...,1.66,2.10,1.68,2.31,1.76,2.35,1.66,2.20,2016_1_brisbane_brisbane_international,2016_1_brisbane_brisbane_international_2016-01...
1,tennis_data_uk,atp,2016,1,Brisbane International,Brisbane,2016-01-04,atp_250,outdoor,hard,...,1.53,2.37,1.63,2.40,1.63,2.50,1.57,2.37,2016_1_brisbane_brisbane_international,2016_1_brisbane_brisbane_international_2016-01...
2,tennis_data_uk,atp,2016,1,Brisbane International,Brisbane,2016-01-04,atp_250,outdoor,hard,...,1.72,2.00,1.90,1.99,1.90,2.10,1.77,2.00,2016_1_brisbane_brisbane_international,2016_1_brisbane_brisbane_international_2016-01...
3,tennis_data_uk,atp,2016,1,Brisbane International,Brisbane,2016-01-04,atp_250,outdoor,hard,...,1.83,1.83,1.93,1.96,1.93,2.10,1.82,1.95,2016_1_brisbane_brisbane_international,2016_1_brisbane_brisbane_international_2016-01...
4,tennis_data_uk,atp,2016,1,Brisbane International,Brisbane,2016-01-05,atp_250,outdoor,hard,...,1.28,3.50,1.31,3.74,1.31,3.80,1.29,3.56,2016_1_brisbane_brisbane_international,2016_1_brisbane_brisbane_international_2016-01...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2621,tennis_data_uk,atp,2016,66,Masters Cup,London,2016-11-18,tour_finals,indoor,hard,...,1.28,3.75,1.31,3.81,1.32,4.10,1.28,3.68,2016_66_london_masters_cup,2016_66_london_masters_cup_2016-11-18_murray_a...
2622,tennis_data_uk,atp,2016,66,Masters Cup,London,2016-11-18,tour_finals,indoor,hard,...,2.37,1.57,2.47,1.61,2.60,1.63,2.41,1.56,2016_66_london_masters_cup,2016_66_london_masters_cup_2016-11-18_cilic_m_...
2623,tennis_data_uk,atp,2016,66,Masters Cup,London,2016-11-19,tour_finals,indoor,hard,...,1.20,4.50,1.24,4.71,1.25,5.00,1.22,4.28,2016_66_london_masters_cup,2016_66_london_masters_cup_2016-11-19_murray_a...
2624,tennis_data_uk,atp,2016,66,Masters Cup,London,2016-11-19,tour_finals,indoor,hard,...,1.22,4.33,1.25,4.55,1.26,5.00,1.23,4.20,2016_66_london_masters_cup,2016_66_london_masters_cup_2016-11-19_djokovic...


In [293]:
csv_path = PROJECT_DIR / f"data/clean/tennis-data-uk/atp/uk_atp_singles_matches_{YEAR}.csv"
csv_path.parent.mkdir(parents=True, exist_ok=True)
df_uk.to_csv(csv_path, index=False)
print(f"Written to {csv_path}")

Written to /Users/nicholasbenelli/Workspace/repos/GitHub/-sports/-tennis/Tennis-Data-Pipeline/data/clean/tennis-data-uk/atp/uk_atp_singles_matches_2016.csv
